This is a simple jupyter notebook that allows you to view the secret deliveries of an archipelago.

It is an **experiment**; I don't know how GitHub's support for Jupyter Notebooks works exactly. Nonetheless, if it works, you are welcome to use it.

Please run the cells one at a time.

In [ ]:
# imports
import ipywidgets as widgets
from IPython.display import display, Markdown
from IPython import display as display_types
from zipfile import ZipFile, BadZipFile
from io import BytesIO
from typing import Optional
from enum import Enum
from pathlib import Path, PathInfo

def manual_slot_name(filename: str, prefix: str) -> str:
  return filename.removeprefix(prefix).removeprefix('_').lstrip('P1234567890').removeprefix('_').removesuffix('.apmanual')

def get_child_data(zipfile: ZipFile, filename: str) -> DisplayObject | None:
  file_ext = Path(filename).suffix
  with zipfile.open(filename, 'r') as subfile_handle:
    file_contents = subfile_handle.read()
  match file_ext:
    case '' | '.txt':
      return display_types.Pretty(file_contents.decode())
    case '.jpg' | '.png' | '.jpeg' | '.gif' | '.webp':
      return display_types.Image(file_contents)
    case '.html' | '.htm':
      return display_types.HTML(file_contents.decode())
    case '.markdown' | '.md':
      return display_types.Markdown(file_contents.decode())
    case '.svg':
      return display_types.SVG(file_contents.decode())
    case _:
      return None

display(Markdown("Please run the next cell."))

In [ ]:
uploader = widgets.FileUpload(
  accept='.zip,.apmanual',
  multiple=False
)

display(Markdown(
'''
Please select either an **.apmanual** file (from the "Download patch file..." option for your player
on the web host) or a **.zip** file (the direct generated output from the AP generator).
'''
))
display(uploader)
display(Markdown(
'''
Once complete, please run the next cell.
'''
))

In [ ]:
if len(uploader.value) == 0:
  display(Markdown(
'''
There are no files selected. Please return to the previous step and select a file.
'''))
  raise SystemExit()

file: ZipFile | None = None
try:
  file = ZipFile(BytesIO(uploader.value[0].content.tobytes()), 'r')
except BadZipFile:
  display(Markdown(
f'''
The file you selected, `{uploader.value[0].name}`, is neither a valid archipelago .zip nor a valid
patch .apmanual. Please return to the previous step and select a different file.
'''))
  raise SystemExit()

display(Markdown(f'File `{uploader.value[0].name}` loaded. Please run the next cell.'))

In [ ]:
# ap generator zip handler
dotarchipelago = [subfile.filename.removesuffix('.archipelago') for subfile in file.filelist if subfile.filename.endswith('.archipelago')]

manual_file: ZipFile | None = None
manual_file_name: str = ''
manual_files: list[str] | None = []

class FileMode(Enum):
  NO_FILE_LOADED = 0
  DIRECT_MANUAL_FILE = 1
  SINGLE_MANUAL_SLOT = 2
  MULTI_MANUAL_SLOTS = 3

file_mode: FileMode = FileMode.NO_FILE_LOADED

if not dotarchipelago:
  if any(subfile.filename == 'archipelago.json' for subfile in file.filelist):
    manual_file = file
    manual_file_name = uploader.value[0].name
    file_mode = FileMode.DIRECT_MANUAL_FILE
    display(Markdown('Please **skip the next cell** and run the cell after it.'))
  else:
    display(Markdown(
f'''
The file you selected, `{uploader.value[0].name}`, is neither a valid archipelago .zip nor a valid
patch .apmanual. Please return to the upload step and select a different file.
'''))
    raise SystemExit()
else:
  prefix = dotarchipelago[0]

  manual_files = [subfile.filename for subfile in file.filelist if subfile.filename.endswith('.apmanual')]

  if len(manual_files) == 0:
    display(Markdown(
f'''
The file you selected, `{uploader.value[0].name}`, has no Manual players within it. Please return to
the upload step and select a different file.
'''
    ))
    raise SystemExit()
  elif len(manual_files) == 1:
    file_mode = FileMode.SINGLE_MANUAL_SLOT
    display(Markdown(
f'''
The file you selected, `{uploader.value[0].name}`, has exactly one Manual slot within it. This slot
will be used. Please **skip the next cell** and run the one after it.
'''
    ))
    manual_file_name = manual_files[0]
    manual_file = ZipFile(file.open(manual_file_name, 'r'), 'r')
  else:
    file_mode = FileMode.MULTI_MANUAL_SLOTS
    display(Markdown(
f'''
The file you selected, `{uploader.value[0].name}`, has multiple Manual slots within it. Please
select one to use, and then run the next cell.
'''
    ))
    filepicker = widgets.Dropdown(
      options=[(manual_slot_name(fname, prefix), fname) for fname in manual_files],
      value=manual_files[0],
      description='Manual slot:',
      disabled=False,
    )
    display(filepicker)

In [ ]:
if file_mode is FileMode.NO_FILE_LOADED:
  display(Markdown(
f'''
<!-- This comment has a width of 100 characters. I want the text to be that wide, too. No wider! -->
You shouldn't be here. No valid file is loaded. Please go back to the file upload cell and upload a
new file.
'''
  ))
  raise SystemExit()
elif file_mode is not FileMode.MULTI_MANUAL_SLOTS:
  display(Markdown(
f'''
This cell is skipped. Please run the next cell.
'''))
else:
  manual_file_name = filepicker.value
  manual_file = ZipFile(file.open(manual_file_name, 'r'), 'r')
  display(Markdown(
f'''
You have selected the manual slot `{manual_slot_name(manual_file_name, prefix)}`. Please run the next cell.
'''
  ))

In [ ]:
if not manual_file:
  display(Markdown(
f'''
You shouldn't be here. No valid file is loaded. Please go back to the file upload cell and upload a
new file.
'''
  ))
  raise SystemExit()

game_data_files = [subfile.filename for subfile in manual_file.filelist if subfile.filename.startswith('game_data/') and not subfile.is_dir()]

display(Markdown(
f'''
<!-- This comment has a width of 100 characters. I want the text to be that wide, too. No wider! -->
These are your game data files. You probably shouldn't open them until your manual game tells you
to! What exactly that means differs from game to game, of course.
'''
))

children: list[DisplayObject] = []
titles: list[str] = []

for subfile in game_data_files:
  if content := get_child_data(manual_file, subfile):
    out = widgets.Output()
    out.append_display_data(content)
    children.append(out)
    titles.append(subfile.removeprefix('game_data/'))

display(widgets.Accordion(
  children=children,
  titles=titles
))